# Transformer Pipeline - 100K Dataset

This notebook fine-tunes `BERT` and `DistilBERT` on `processed_data/dataset_100k`.

The execution is split into three parts:

1. Run `BERT` and save its outputs immediately.
2. Run `DistilBERT` and save its outputs immediately.
3. Combine both model summaries into one final comparison table.

- `metrics.json`
- `training_history.csv`
- `classification_report.csv`
- `confusion_matrix.csv`
- `confusion_matrix.png`
- `test_predictions.csv`
- optional attention visualization outputs

The random seed is fixed at `42` to match your team plan.


In [ ]:
import subprocess
subprocess.run(["pip", "install", "sentencepiece"])
subprocess.run(["pip", "uninstall", "torch", "torchvision", "torchaudio", "-y"])

subprocess.run(["pip", "install", "torch", "torchvision", "torchaudio", 
                "--index-url", "https://download.pytorch.org/whl/cu121"])

packages = [
    "pandas",
    "numpy", 
    "transformers",
    "scikit-learn",
    "matplotlib",
    "ipywidgets",
    "transformers-interpret"
]
for pkg in packages:
    subprocess.run(["pip", "install", pkg])

print("All packages installed successfully!")

Looking in indexes: http://mirrors.aliyun.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 9.6 MB/s eta 0:00:00:00:0100:01


Found existing installation: torch 2.3.0+cu121
Uninstalling torch-2.3.0+cu121:
  Successfully uninstalled torch-2.3.0+cu121
Found existing installation: torchvision 0.18.0+cu121
Uninstalling torchvision-0.18.0+cu121:
  Successfully uninstalled torchvision-0.18.0+cu121


Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 4.4 MB/s eta 0:00:0000:0100:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 2.7 MB/s eta 0:00:0000:0100:010m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 10.9 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 4.4 MB/s eta 0:00:0000:0100:03
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 7.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.6/209.6 MB 4.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 3.4 MB/s eta 0:00:00a 0:00:010m
  Attempting uninstall: sympy
    Found existing installation: sympy 1.12.1
    Uninstalling sympy-1.12.1:
      Successfully uninstalled sympy-1.12.1
  Attempting uninstall: nvidia-nccl-cu12
    Found existing installation: nvidia-nccl-cu12 2.20.5
    Uninstalling nvidia-nccl-cu12-2.20.5:
      Successfully un

Looking in indexes: http://mirrors.aliyun.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 17.1 MB/s eta 0:00:0000:0100:01


Looking in indexes: http://mirrors.aliyun.com/pypi/simple


Looking in indexes: http://mirrors.aliyun.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 25.1 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 63.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.1/801.1 kB 17.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 21.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 26.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 24.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 41.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 42.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 7.2 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.3/87.3 kB 2.6 MB/s eta 0:00:00


Looking in indexes: http://mirrors.aliyun.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 18.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 10.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 4.9 MB/s eta 0:00:0000:01


Looking in indexes: http://mirrors.aliyun.com/pypi/simple


Looking in indexes: http://mirrors.aliyun.com/pypi/simple


Looking in indexes: http://mirrors.aliyun.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 3.9 MB/s eta 0:00:00a 0:00:010m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.8/793.8 kB 7.7 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: ipython
    Found existing installation: ipython 8.25.0
    Uninstalling ipython-8.25.0:
      Successfully uninstalled ipython-8.25.0
All packages installed successfully!


In [1]:
from pathlib import Path
PROJECT_ROOT = Path("/root/autodl-tmp")
DATASET_NAME = "dataset_100k"
DATASET_DIR = Path("processed_data") / DATASET_NAME
OUTPUT_ROOT = Path("member_c_outputs")
BERT_MODEL_KEY = "bert"
DISTILBERT_MODEL_KEY = "distilbert"
EPOCHS = 3
BATCH_SIZE = 8
MAX_LENGTH = 256
LEARNING_RATE = 2e-5
SEED = 42
ATTENTION_SAMPLES = 3
SKIP_ATTENTION = False
DATASET_DIR


PosixPath('processed_data/dataset_100k')

In [2]:
import json
import os
import random
import time
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str((Path.cwd() / ".matplotlib_cache").resolve()))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix, f1_score
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

LABEL_NAMES = {0: "Human", 1: "AI"}
MODEL_MAP = {
    "bert": "/root/autodl-tmp/bert-base-uncased",
    "distilbert": "/root/autodl-tmp/distilbert-base-uncased",
}


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def read_split(csv_path: Path) -> pd.DataFrame:
    dataframe = pd.read_csv(csv_path)
    required_columns = {"text", "generated"}
    missing_columns = required_columns.difference(dataframe.columns)
    if missing_columns:
        raise ValueError(f"{csv_path} is missing columns: {sorted(missing_columns)}")
    dataframe = dataframe.dropna(subset=["text", "generated"]).copy()
    dataframe["text"] = dataframe["text"].astype(str)
    dataframe["generated"] = dataframe["generated"].astype(int)
    return dataframe.reset_index(drop=True)


class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = self.texts[index]
        label = self.labels[index]
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_attention_mask=True,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long),
        }


def create_data_loader(dataframe, tokenizer, max_length, batch_size, shuffle, device):
    dataset = TextClassificationDataset(
        texts=dataframe["text"].tolist(),
        labels=dataframe["generated"].tolist(),
        tokenizer=tokenizer,
        max_length=max_length,
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=device.type == "cuda",
    )


def train_epoch(model, data_loader, optimizer, scheduler, device):
    model.train()
    losses = []
    predictions = []
    true_labels = []

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        losses.append(loss.item())
        predictions.extend(preds.detach().cpu().tolist())
        true_labels.extend(labels.detach().cpu().tolist())

    return {
        "loss": float(np.mean(losses)) if losses else 0.0,
        "accuracy": accuracy_score(true_labels, predictions),
        "macro_f1": f1_score(true_labels, predictions, average="macro", zero_division=0),
    }


def evaluate_model(model, data_loader, device):
    model.eval()
    losses = []
    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1)

            losses.append(loss.item())
            predictions.extend(preds.detach().cpu().tolist())
            true_labels.extend(labels.detach().cpu().tolist())

    return {
        "loss": float(np.mean(losses)) if losses else 0.0,
        "accuracy": accuracy_score(true_labels, predictions),
        "macro_f1": f1_score(true_labels, predictions, average="macro", zero_division=0),
        "predictions": predictions,
        "true_labels": true_labels,
    }


def save_confusion_outputs(y_true, y_pred, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    class_order = sorted(LABEL_NAMES)
    display_labels = [LABEL_NAMES[label_id] for label_id in class_order]
    matrix = confusion_matrix(y_true, y_pred, labels=class_order)
    pd.DataFrame(matrix, index=display_labels, columns=display_labels).to_csv(output_dir / "confusion_matrix.csv")

    figure, axis = plt.subplots(figsize=(6, 6))
    ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=display_labels).plot(
        ax=axis,
        cmap="Blues",
        colorbar=False,
    )
    figure.tight_layout()
    figure.savefig(output_dir / "confusion_matrix.png", dpi=200, bbox_inches="tight")
    plt.close(figure)


def save_attention_visualizations(model, tokenizer, predictions_dataframe, output_dir, sample_count):
    output_dir.mkdir(parents=True, exist_ok=True)
    status_file = output_dir / "attention_visualization_status.txt"

    try:
        from transformers_interpret import SequenceClassificationExplainer
    except Exception as exc:
        status_file.write_text(
            "Skipped attention visualization because transformers-interpret is not available.\n"
            "Install with: pip install transformers-interpret\n"
            f"Original import error: {exc}\n",
            encoding="utf-8",
        )
        return

    sample_dataframe = predictions_dataframe.copy()
    sample_dataframe["correct"] = sample_dataframe["generated"] == sample_dataframe["prediction"]
    sample_dataframe = sample_dataframe.sort_values(by=["correct", "text"], ascending=[True, True]).head(sample_count)

    if sample_dataframe.empty:
        status_file.write_text("No samples available for attention visualization.\n", encoding="utf-8")
        return

    explainer = SequenceClassificationExplainer(model, tokenizer)
    summary_lines = []

    for sample_index, row in enumerate(sample_dataframe.itertuples(index=False), start=1):
        text = str(row.text)
        tokens = tokenizer.encode(text)
        if len(tokens) > 510:
            tokens = tokens[:510]
            text = tokenizer.decode(tokens, skip_special_tokens=True)
        attributions = explainer(text)
        attribution_dataframe = pd.DataFrame(attributions, columns=["token", "attribution"])
        attribution_dataframe = attribution_dataframe[attribution_dataframe["token"].astype(str).str.strip().ne("")].copy()
        attribution_dataframe["abs_attribution"] = attribution_dataframe["attribution"].abs()
        attribution_dataframe.to_csv(output_dir / f"sample_{sample_index:02d}_token_attributions.csv", index=False)

        top_tokens = attribution_dataframe.nlargest(12, "abs_attribution").iloc[::-1]
        figure, axis = plt.subplots(figsize=(10, 5))
        axis.barh(top_tokens["token"], top_tokens["attribution"], color="#3b82f6")
        axis.set_title(
            f"Sample {sample_index}: true={LABEL_NAMES[int(row.generated)]}, pred={LABEL_NAMES[int(row.prediction)]}"
        )
        axis.set_xlabel("Token attribution")
        figure.tight_layout()
        figure.savefig(output_dir / f"sample_{sample_index:02d}_top_tokens.png", dpi=200, bbox_inches="tight")
        plt.close(figure)

        top_token_text = ", ".join(top_tokens["token"].astype(str).tolist())
        summary_lines.append(
            f"sample_{sample_index:02d}: true={LABEL_NAMES[int(row.generated)]}, pred={LABEL_NAMES[int(row.prediction)]}, top_tokens=[{top_token_text}]"
        )

    status_file.write_text("\n".join(summary_lines) + "\n", encoding="utf-8")


def sanitize_model_name(model_name):
    return model_name.replace("/", "_").replace("-", "_")


def run_single_model(dataset_name, dataset_dir, output_root, model_key, epochs, batch_size, max_length, learning_rate, seed, attention_samples, skip_attention):
    set_seed(seed)
    device = get_device()
    model_name = MODEL_MAP[model_key]
    model_slug = sanitize_model_name(model_name)
    model_output_dir = output_root / dataset_name / model_slug
    model_output_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n=== Dataset: {dataset_name} | Model: {model_name} ===")
    print(f"Using device: {device}")

    train_dataframe = read_split(dataset_dir / "train.csv")
    validation_dataframe = read_split(dataset_dir / "validation.csv")
    test_dataframe = read_split(dataset_dir / "test.csv")

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        id2label=LABEL_NAMES,
        label2id={label_name: label_id for label_id, label_name in LABEL_NAMES.items()},
    ).to(device)

    train_loader = create_data_loader(train_dataframe, tokenizer, max_length, batch_size, True, device)
    validation_loader = create_data_loader(validation_dataframe, tokenizer, max_length, batch_size, False, device)
    test_loader = create_data_loader(test_dataframe, tokenizer, max_length, batch_size, False, device)

    optimizer = AdamW(model.parameters(), lr=learning_rate)
    total_training_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_training_steps,
    )

    training_history = []
    best_validation_f1 = -1.0
    best_model_dir = model_output_dir / "best_model"

    start_time = time.perf_counter()
    for epoch_index in range(1, epochs + 1):
        train_metrics = train_epoch(model, train_loader, optimizer, scheduler, device)
        validation_metrics = evaluate_model(model, validation_loader, device)

        epoch_metrics = {
            "epoch": epoch_index,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "val_loss": validation_metrics["loss"],
            "val_accuracy": validation_metrics["accuracy"],
            "val_macro_f1": validation_metrics["macro_f1"],
        }
        training_history.append(epoch_metrics)

        print(
            f"Epoch {epoch_index}/{epochs} | "
            f"train_loss={epoch_metrics['train_loss']:.4f} | "
            f"train_acc={epoch_metrics['train_accuracy']:.4f} | "
            f"val_loss={epoch_metrics['val_loss']:.4f} | "
            f"val_macro_f1={epoch_metrics['val_macro_f1']:.4f}"
        )

        if validation_metrics["macro_f1"] > best_validation_f1:
            best_validation_f1 = float(validation_metrics["macro_f1"])
            model.save_pretrained(best_model_dir)
            tokenizer.save_pretrained(best_model_dir)

    training_seconds = time.perf_counter() - start_time
    pd.DataFrame(training_history).to_csv(model_output_dir / "training_history.csv", index=False)

    best_model = AutoModelForSequenceClassification.from_pretrained(best_model_dir).to(device)
    test_metrics = evaluate_model(best_model, test_loader, device)
    y_true = list(test_metrics["true_labels"])
    y_pred = list(test_metrics["predictions"])

    report_dataframe = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=sorted(LABEL_NAMES),
            target_names=[LABEL_NAMES[label_id] for label_id in sorted(LABEL_NAMES)],
            output_dict=True,
            zero_division=0,
        )
    ).transpose()
    report_dataframe.to_csv(model_output_dir / "classification_report.csv")

    predictions_dataframe = test_dataframe.copy()
    predictions_dataframe["prediction"] = y_pred
    predictions_dataframe["true_label"] = predictions_dataframe["generated"].map(LABEL_NAMES)
    predictions_dataframe["predicted_label"] = predictions_dataframe["prediction"].map(LABEL_NAMES)
    predictions_dataframe["is_correct"] = predictions_dataframe["generated"] == predictions_dataframe["prediction"]
    predictions_dataframe.to_csv(model_output_dir / "test_predictions.csv", index=False)

    save_confusion_outputs(y_true, y_pred, model_output_dir)

    if skip_attention:
        (model_output_dir / "attention_visualization_status.txt").write_text(
            "Skipped attention visualization because SKIP_ATTENTION is True.\n",
            encoding="utf-8",
        )
    else:
        save_attention_visualizations(
            model=best_model,
            tokenizer=tokenizer,
            predictions_dataframe=predictions_dataframe,
            output_dir=model_output_dir / "attention_visualizations",
            sample_count=attention_samples,
        )

    summary = {
        "dataset": dataset_name,
        "dataset_dir": str(dataset_dir),
        "model_key": model_key,
        "model_name": model_name,
        "device": str(device),
        "seed": seed,
        "epochs": epochs,
        "batch_size": batch_size,
        "max_length": max_length,
        "learning_rate": learning_rate,
        "training_seconds": round(training_seconds, 4),
        "test_loss": round(float(test_metrics["loss"]), 6),
        "test_accuracy": round(float(test_metrics["accuracy"]), 6),
        "test_macro_f1": round(float(test_metrics["macro_f1"]), 6),
    }

    with (model_output_dir / "metrics.json").open("w", encoding="utf-8") as output_file:
        json.dump(summary, output_file, indent=2)

    print(
        f"Finished {dataset_name} / {model_name} | "
        f"test_accuracy={summary['test_accuracy']:.4f} | "
        f"test_macro_f1={summary['test_macro_f1']:.4f}"
    )

    return summary


def run_all_models(dataset_name, dataset_dir, output_root, models, epochs, batch_size, max_length, learning_rate, seed, attention_samples, skip_attention):
    summaries = []
    for model_key in models:
        summaries.append(
            run_single_model(
                dataset_name=dataset_name,
                dataset_dir=dataset_dir,
                output_root=output_root,
                model_key=model_key,
                epochs=epochs,
                batch_size=batch_size,
                max_length=max_length,
                learning_rate=learning_rate,
                seed=seed,
                attention_samples=attention_samples,
                skip_attention=skip_attention,
            )
        )
    return pd.DataFrame(summaries)


## Part 1 - Run BERT

Run this cell first if you want the `BERT` result immediately.


In [4]:
bert_result = run_single_model(
    dataset_name=DATASET_NAME,
    dataset_dir=DATASET_DIR,
    output_root=OUTPUT_ROOT,
    model_key=BERT_MODEL_KEY,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    learning_rate=LEARNING_RATE,
    seed=SEED,
    attention_samples=ATTENTION_SAMPLES,
    skip_attention=SKIP_ATTENTION,
)

bert_result_df = pd.DataFrame([bert_result])
bert_result_df



=== Dataset: dataset_100k | Model: /root/autodl-tmp/bert-base-uncased ===
Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: /root/autodl-tmp/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3 | train_loss=0.0492 | train_acc=0.9891 | val_loss=0.0425 | val_macro_f1=0.9933


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch 3/3 | train_loss=0.0027 | train_acc=0.9995 | val_loss=0.0228 | val_macro_f1=0.9963


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (566 > 512). Running this sequence through the model will result in indexing errors


Finished dataset_100k / /root/autodl-tmp/bert-base-uncased | test_accuracy=0.9978 | test_macro_f1=0.9978


,dataset,dataset_dir,model_key,model_name,device,seed,epochs,batch_size,max_length,learning_rate,training_seconds,test_loss,test_accuracy,test_macro_f1
0,dataset_100k,processed_data/dataset_100k,bert,/root/autodl-tmp/bert-base-uncased,cuda,42,3,8,256,0.00002,2153.9359,0.013189,0.9978,0.9978


## Part 2 - Run DistilBERT

Run this cell after `BERT`. It will save the `DistilBERT` outputs independently.


In [3]:
distilbert_result = run_single_model(
    dataset_name=DATASET_NAME,
    dataset_dir=DATASET_DIR,
    output_root=OUTPUT_ROOT,
    model_key=DISTILBERT_MODEL_KEY,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    learning_rate=LEARNING_RATE,
    seed=SEED,
    attention_samples=ATTENTION_SAMPLES,
    skip_attention=SKIP_ATTENTION,
)

distilbert_result_df = pd.DataFrame([distilbert_result])
distilbert_result_df



=== Dataset: dataset_100k | Model: /root/autodl-tmp/distilbert-base-uncased ===
Using device: cuda


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: /root/autodl-tmp/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3 | train_loss=0.0549 | train_acc=0.9866 | val_loss=0.0454 | val_macro_f1=0.9920


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch 3/3 | train_loss=0.0030 | train_acc=0.9994 | val_loss=0.0154 | val_macro_f1=0.9976


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Finished dataset_100k / /root/autodl-tmp/distilbert-base-uncased | test_accuracy=0.9975 | test_macro_f1=0.9975


,dataset,dataset_dir,model_key,model_name,device,seed,epochs,batch_size,max_length,learning_rate,training_seconds,test_loss,test_accuracy,test_macro_f1
0,dataset_100k,processed_data/dataset_100k,distilbert,/root/autodl-tmp/distilbert-base-uncased,cuda,42,3,8,256,0.00002,1353.5298,0.015469,0.9975,0.9975


## Part 3 - Final Summary

Run this cell after both models finish to get one combined comparison table.


In [4]:
summary_rows = []

if 'bert_result' in globals():
    summary_rows.append(bert_result)

if 'distilbert_result' in globals():
    summary_rows.append(distilbert_result)

if not summary_rows:
    raise ValueError('No model results found yet. Run the BERT and/or DistilBERT cells first.')

final_results_df = pd.DataFrame(summary_rows)
final_results_df = final_results_df.sort_values(by='model_name').reset_index(drop=True)
final_results_df.to_csv(OUTPUT_ROOT / DATASET_NAME / 'combined_model_summary.csv', index=False)
final_results_df


,dataset,dataset_dir,model_key,model_name,device,seed,epochs,batch_size,max_length,learning_rate,training_seconds,test_loss,test_accuracy,test_macro_f1
0,dataset_100k,processed_data/dataset_100k,distilbert,/root/autodl-tmp/distilbert-base-uncased,cuda,42,3,8,256,0.00002,1353.5298,0.015469,0.9975,0.9975
